In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_csv("../data/Base.csv")

In [3]:
df = df.drop(columns=["device_fraud_count"])

In [4]:
mis_cols = ["prev_address_months_count", "current_address_months_count",
                "bank_months_count", "session_length_in_minutes",
                "device_distinct_emails_8w"]
for col in mis_cols:
    df[col] = df[col].replace(-1, np.nan)

cat_cols = ["payment_type", "employment_status", "housing_status", "source", "device_os"]
df = pd.get_dummies(df, columns=cat_cols, dtype=int)

In [5]:
X = df.drop(columns=["fraud_bool" , "month"])
y = df["fraud_bool"].values

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4,stratify=y, random_state=42)

X_val , X_test , y_val , y_test = train_test_split(X_temp, y_temp, test_size=0.5,stratify=y_temp, random_state=42)

print("Rows: ",len(X_train), len(X_val), len(X_test))
print("Fraud rate: ",y_train.mean(), y_val.mean(), y_test.mean())

Rows:  600000 200000 200000
Fraud rate:  0.011028333333333333 0.01103 0.01103


In [6]:
# filling missing values with median

for col in mis_cols:
    median = X_train[col].median()
    X_train[col] = X_train[col].fillna(median)
    X_val[col] = X_val[col].fillna(median)
    X_test[col] = X_test[col].fillna(median)

older_val = (X_val["customer_age"] >= 50).values
older_test = (X_test["customer_age"] >= 50).values

#scaling the data

feat_names = list(X_train.columns)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s = scaler.transform(X_val)
X_test_s = scaler.transform(X_test)

print("Baseline accuracy :", round(1-y_train.mean(), 4))


Baseline accuracy : 0.989


In [7]:
from sklearn.metrics import precision_score, recall_score, f1_score

def evaluate(name, y_true, y_pred, older):
    honest = (y_true == 0)
    fpr_older = y_pred[honest & older].mean()
    fpr_younger = y_pred[honest & ~older].mean() #here ~older means younger 
    return {"model":name,
            "precision": precision_score(y_true, y_pred),
            "recall": recall_score(y_true, y_pred),
            "f1": f1_score(y_true, y_pred),
            "fpr_older": fpr_older,
            "fpr_younger": fpr_younger,
            "fpr_ratio":min(fpr_older, fpr_younger)/max(fpr_older, fpr_younger)
            }


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB

models = {
    "Logistic Regression": LogisticRegression(class_weight="balanced", max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, class_weight="balanced", random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=50, max_depth=10, min_samples_leaf=50,
                                            class_weight="balanced", n_jobs=-1, random_state=42),
    "Naive Bayes": GaussianNB(),
}

val_results = []
for name, model in models.items():
    model.fit(X_train_s, y_train)
    val_results.append(evaluate(name, y_val, model.predict(X_val_s), older_val))

pd.DataFrame(val_results).round(3)

,model,precision,recall,f1,fpr_older,fpr_younger,fpr_ratio
0,Logistic Regression,0.043,0.790,0.081,0.365,0.161,0.440
1,Decision Tree,0.037,0.731,0.070,0.307,0.192,0.624
2,Random Forest,0.051,0.747,0.096,0.291,0.124,0.426
3,Naive Bayes,0.015,0.941,0.030,0.804,0.638,0.794


In [11]:
test_results = []
for name, model in models.items():
    test_results.append(evaluate(name, y_test, model.predict(X_test_s), older_test))

pd.DataFrame(test_results).round(3)

,model,precision,recall,f1,fpr_older,fpr_younger,fpr_ratio
0,Logistic Regression,0.043,0.787,0.081,0.364,0.159,0.436
1,Decision Tree,0.036,0.712,0.068,0.308,0.193,0.627
2,Random Forest,0.050,0.738,0.094,0.297,0.125,0.422
3,Naive Bayes,0.016,0.943,0.031,0.802,0.637,0.794
